#Covered Today:

1. Tokenizers: How LLMs Convert Text to Numbers
2. Tokenizers in Action: Encoding and Decoding with Llama 3.1
3. How Chat Templates Work: LLaMA Tokenizers and Special Tokens
4. Comparing Tokenizers: Phi-4, DeepSeek, and QWENCoder in Action

In [20]:
# we first start with a basic understanding of tokenzier. A tokenizer works in two steps. It starts by first breaking down the given text into tokens. Basically, breaks natural langauge into chunks (blocks of letters) and then these tokens are assigned a number.

# This number that is assigned to these tokens is called token IDs.

# So token is the chunk of letters whereas token id is the number assigned to that chunk of letter. These two are used interchangebly, mostly when people say token, they are referring to token id.

# The tokenizer contains a dictionary, where all the possible tokens for that model reside, and the lookup value is the number, the token id. This dictionary also contains some special tokens that do not match the natural language. These tokens are for providing special information to the LLM. Such as a specific token ID or number that highlights the start of the prompt.

# For example, we assigned number 10 to the start of the prompt. Now, for every prompt that was sent to the LLM/neural network, during the entire training period, had 10 at the beginning. So, with repeated signalling, by sending in 10, neural network understood that 10 means the beginning of the prompt, similarly how it became good at producing natural language.

# Also, different LLMs do not use the same tokenizer, when someone goes ahead and starts training a LLM, they first start by building their own tokenizer.

# Now: how tokens are decided? First before we start training the tokenizer, we give it a fix value in terms of total vocab. Then we dump in the text, which contains all types of texts that we want the LLM to be good at, example: code, story, general chat discussion etc. Now, the tokenizer breaks down the text into smallest possible units (all individual raw characters or standard byte-level values (0–255)). This ensures that entire vocab is accounted for, and never ever we encounter any error in identification of a character.

# This is followed by counting of the pair frequencies, such as how many times t & h, or i and n or e and r are being used together. The most common of such pairs are then merged into a single new token. So, th now becomes one token.

# This continues with more token pairs, such as th and e, becomes the, similary tokens are then created on the basis of their frequency in the input text. It stops, when the given vocab count is reached.

In [21]:
# Now, like last time, we only install any libraries or versions if code does not work, so that we also understand the difference between the versions.

# We start by loggin in to hugging face and print out the GPU details
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [22]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)

if gpu_info.find("Tesla T4") >= 1 and gpu_info.find("CUDA Version") >= 1:
  print("Connected to Nvidia TESLA T4 GPU")
  print(gpu_info)
else:
  print("Please check, GPU not connected")

Connected to Nvidia TESLA T4 GPU
Wed Aug 26 06:32:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------

In [23]:
# To use tokenizers present on hugging face, we need to import: from transformers import AutoTokenizer. Now, transformers is a library created by hugging face which is used to work with pretrained models. AutoTokenizer is the class under this library, which is used to find the appropriate tokenizer for a given model, so no need to find, it does the job by iteself. I am putting this above in the first cell. and then re-running the cells.

# Let's go ahead and create a tokenizer now:
# to create a tokenizer, we save the value of this syntax in a variable:
# AutoTokenizer.from_pretrained("model-name")
# here, by specifying from_pretrained, we are telling huggingface that we need to fetch the tokenizer, from your pretrained(with it's vocab and other info) set of tokenizers, which is meant for the model name, which we pass in as the argument.

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", trust_remote_code=True)

# one more point to be noted here is the use of trust_remote_code. Now, there are two types of models on the basis of implementation architecture. In case 1, everything needed to run the model is present in the transformers library provided by huggingface. However, let's say if there is a new model launched, with a different underlying architecture, then it might accompany some more code needed to properly run the model. In this case, we need to specify it to the transformers library whether we can trust the code accompanying this model or not. In absence of this parameter, it always remains False.

In [24]:
text = "I am learning LLM Engineering"
# now, we can use the tokenizer that we had created above, to convert any given text into token ids based on the tokenizer the creators of that model have used.

# Also, the process of converting given text to token_ids is called encoding.

token_ids = tokenizer.encode(text)
print(token_ids)

[128000, 40, 1097, 6975, 445, 11237, 17005]


In [25]:
# our sentence had 5 words, however, the count of output tokens is 7.

In [26]:
# now, we have a list of token_ids based on the text that we had given it. We can now further use this list to convert given token_ids to text by decoding.

id_to_text = tokenizer.decode(token_ids)
print(id_to_text)

<|begin_of_text|>I am learning LLM Engineering


In [27]:
# we go a warning while decoding to natural language and for that system asks us to use: clean_up_tokenization_spaces=False
id_to_text = tokenizer.decode(token_ids, clean_up_tokenization_spaces=False)
print(id_to_text)

<|begin_of_text|>I am learning LLM Engineering


In [28]:
# Now we can clearly see the output. Now, as noticed, one token has been used to highlighting the beginning of text also. which from the list of token_ids seens to be 128000, lets decode this alone

tokenizer.decode(128000)

'<|begin_of_text|>'

In [29]:
# yes, so 128000 is the token ID assigned to token: <|begin_of_text|>
# also, in addition to running decode, we can also use batch_encode, which will give us a list of all tokens in a list associated with the given input token_ids, lets try it out also:
token_ids = tokenizer.encode(text)
print(token_ids)
tokenizer.batch_decode(token_ids)


[128000, 40, 1097, 6975, 445, 11237, 17005]


['<|begin_of_text|>I am learning LLM Engineering']

In [30]:
# we can see that we are not getting the exact required output and this is basically due to the transformer version that we are using, if we want to now print out each individual token from its token_id, we do the following:
[tokenizer.batch_decode(x) for x in token_ids]

[['<|begin_of_text|>'],
 ['I'],
 [' am'],
 [' learning'],
 [' L'],
 ['LM'],
 [' Engineering']]

In [31]:
# this will give us each of the individual tokens.

# Moving on, so far we were using meta-llama/Llama-3.1-8B, now this is a base model, meaning this has not been trained to follow the chat format(system, user etc.). This base model can be taken up and can be further trained.

# In huggingface, models which are meant for chat interface or basically are trained for that format, have either instruct or chat in their names. The chat version of this model is called: meta-llama/Llama-3.1-8B-Instruct.

# Now, there is a specific format that the LLM expects, so that it can respond to us. Here, we need to remember that LLM just predicts the next word, and the input it expects to do that is something like this:
# <start><system><system_prompt><end><start><user><user_prompt><end><start><assistant>

# Now, while the above is not exactly how it behaves, we can understand that we have give it a system prompt and a user prompt and left the assistant section without a prompt or ending, by this a model will understand that not it has to predict what an assistant would say in this situation.

# We have been working with a dict style input system so far and huggingface have given us a way to convert that input system into a system that is understood by the LLMs as in the above format. To apply that system, we run apply chat template, lets have a look at it below:

# we start by creating a tokenizer now for the instruct model

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", trust_remote_code=True)

In [32]:
# since this tokenizer is different for this model, hence this was downloaded again. Next up, lets create a message dict input template, that we are accustomed to:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell us a light hearted joke about a software engineer"}
]

# this is what we have been used to, however, we now convert this into LLM understandable format
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

# now, in the above, we also have two more parameters, these are 1. tokenizer: it will directly convert the given messages into token ids, which we would not be able to read and remains True by default, therefore we need to specify, that even though you apply the chat template, but it should be human readable, not a bunch of numbers, while add_generation_prompt is to inform the tokenizer that this input is going to be an input for the chat interface, so add the assistant info at the end. We experiment with both these next.

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell us a light hearted joke about a software engineer<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [33]:
# now as we can see the prompt, we have some details:
# 1.<|begin_of_text|><|start_header_id|>system<|end_header_id|>, here we first see begin of text, meaning text is beginning from here, then we have start_header_id > header Ids are given to system, user, assistant or tool to help with identification of the roles. therefore, start header id and end header id.
# next we have cutoff knowledge date: of the model and today date is the date given by the model provider to give a context of time to the LLM, since it has no internal clock.
# Then we have system prompt, followed by user role and its prompt and finally, assistant but no prompt, this is now telling the LLM, to predict what the assistant would do next, making this a very reliable outcome from the chat that we can have with an LLM.
# finally, there's also eot_id: which means end of turn. Now, with time, the LLM during a chat, will have multiple user, system or tool sections in the context, then it becomes important to tell the LLM that okay, here is the end of turn of the user and next header begins.


# let's now check the output when we do not use any of the two other parameters:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell us a light hearted joke about a software engineer<|eot_id|>


In [34]:
# In the above case, no assistant header is added to the prompt
prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
print(prompt)

{'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 2675, 527, 264, 11190, 18328, 128009, 128006, 882, 128007, 271, 41551, 603, 264, 3177, 4851, 291, 22380, 922, 264, 3241, 24490, 128009, 128006, 78191, 128007, 271], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [36]:
# now we only get token IDs.
# Also one more point to be added here is that the training data was something in this format:
# <start><system><system_prompt><end><start><user><user_prompt><end><start><assistant><assistant_prompt><end>
# so by feeding this in the neural netword thousands of times, we told the model how to respond when <assistant> responds, and it actually mimics what it was trained on.

# next, we do this with multiple models
# lets create their individual tokenizers first

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
deep_seek_tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")
phi4_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

In [42]:
text = "I am excited to show how different tokenizers from different LLMs work. This is great"
print("LLama")
token_ids = tokenizer.encode(text)
print(token_ids)
tokens = [tokenizer.decode(x) for x in token_ids]
print(tokens)

print("\n\n")
print("Qwen")
token_ids = qwen_tokenizer.encode(text)
print(token_ids)
tokens = [qwen_tokenizer.decode(x) for x in token_ids]
print(tokens)

print("\n\n")
print("DeepSeek")
token_ids = deep_seek_tokenizer.encode(text)
print(token_ids)
tokens = [deep_seek_tokenizer.decode(x) for x in token_ids]
print(tokens)

print("\n\n")
print("Phi4")
token_ids = phi4_tokenizer.encode(text)
print(token_ids)
tokens = [phi4_tokenizer.decode(x) for x in token_ids]
print(tokens)

LLama
[128000, 40, 1097, 12304, 311, 1501, 1268, 2204, 4037, 12509, 505, 2204, 445, 11237, 82, 990, 13, 1115, 374, 2294]
['<|begin_of_text|>', 'I', ' am', ' excited', ' to', ' show', ' how', ' different', ' token', 'izers', ' from', ' different', ' L', 'LM', 's', ' work', '.', ' This', ' is', ' great']



Qwen
[40, 1079, 12035, 311, 1473, 1246, 2155, 3950, 12230, 504, 2155, 444, 10994, 82, 975, 13, 1096, 374, 2244]
['I', ' am', ' excited', ' to', ' show', ' how', ' different', ' token', 'izers', ' from', ' different', ' L', 'LM', 's', ' work', '.', ' This', ' is', ' great']



DeepSeek
[40, 608, 9216, 276, 1296, 946, 1448, 10728, 18845, 473, 1448, 27344, 19821, 830, 13, 1002, 317, 1228]
['I', ' am', ' excited', ' to', ' show', ' how', ' different', ' token', 'izers', ' from', ' different', ' LL', 'Ms', ' work', '.', ' This', ' is', ' great']



Phi4
[40, 939, 15209, 316, 2356, 1495, 2647, 6602, 24223, 591, 2647, 451, 19641, 82, 1101, 13, 1328, 382, 2212]
['I', ' am', ' excited', ' to',

In [45]:
# all the numbers are different, and in the other 3 models, no begin of text exists. Lets now apply chat template to all of these.
print("LLama")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n")
print("Qwen")
print(qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n")
print("DeepSeek")
print(deep_seek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n")
print("Phi4")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

LLama
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell us a light hearted joke about a software engineer<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Qwen
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Tell us a light hearted joke about a software engineer<|im_end|>
<|im_start|>assistant



DeepSeek
<｜begin▁of▁sentence｜>You are a helpful assistant

User: Tell us a light hearted joke about a software engineer

Assistant:


Phi4
<|system|>You are a helpful assistant<|end|><|user|>Tell us a light hearted joke about a software engineer<|end|><|assistant|>


In [ ]:
# While they have different formats, but if we look closely, all of them follow the same pattern.